# How much data is there and what does one example look like?

**Goal:**
finding out what I'm actually working with before building anything.

**How to think about it:**
pretend someone handed you this file and said "build a model on it by Friday". First asks: what is this, and can I trust it? Datasets from the internet are never as clean as the description says. The job here is to find out where it differs from the claims.

# How much data is there, and what does one example look like?

In [ ]:
import numpy as np
DEFECTS = ["Center", "Donut", "Edge-Loc", "Edge-Ring", "Loc", "Near-Full", "Scratch", "Random"]

data = np.load("../data/Wafer_Map_Datasets.npz")
maps = data["arr_0"]    # wafer maps
labels = data["arr_1"]  # labels

#(count, width, height, data type)
#(38015, 52, 52, int32)
print("Map:  ", maps.shape, labels.dtype)

#(count, columns, data type)
#(38015, 8, int32)
print("labels:", labels.shape, labels.dtype)

# What do the numbers in a map mean?
Across all 38,015 wafers combined:

Value 0 (background/no die): 24M pixels: empty space on the wafer\
Value 1 (good die): 53M pixels: working silicon, ~50% of all pixels\
Value 2 (failed die): 25M pixels: defective silicon, ~25% of all pixels\
Value 3 (anomaly): 214 pixels: noise or labeling error, negligible

In [ ]:
values, counts = np.unique(maps, return_counts=True)

for v, c in zip(values, counts):
    print(f"wafer {v}: {c:,} pixels")


# What does the label look like, and what does it mean?


In [ ]:
def label_name(label):
    defect_present = []
    # Loop through each defect position (0-7)
    for i in range(len(DEFECTS)):
        # If this position is 1, the defect is present
        if label[i] == 1:
            defect_present.append(DEFECTS[i])
    return " + ".join(defect_present)


In [ ]:
# The column order comes from the dataset description.
# DEFECTS is defined here, but the actual label values come from the dataset.
# A label: [0, 1, 0, 0, 1, 0, 0, 0] a binary vector where 1 means "this defect is present"
# random_indices exists because the wafers are sorted, and by only loading 10 we wouldn't see any differences in the prints

random_indices = np.random.choice(len(labels), 10, replace=False)
for i in random_indices:
    print(f"{labels[i]} - {label_name(labels[i])}")


# How many wafers per defect type? Are some defects common, others rare?

In [ ]:
for i, defect in enumerate(DEFECTS):
    for j in range(len(labels[:, i])):
        column = labels[:, i]
        has_defect = (column != 1)
        count = has_defect.sum()
    print(f"{defect}: {count:,} wafers")

# Is anything missing, strange, or there twice?

In [334]:
# What are the min/max pixel values?
min_pixel = maps.min()
max_pixel = maps.max()
# print(f"min: {min_pixel}, max: {max_pixel}")

# What are the min/max label values?
min_label = labels.min()
max_label = labels.max()
# print(f"min: {min_label}, max: {max_label}")

# How many NaNs exist?
nan_count = np.isnan(maps.astype(float)).sum()
# print(f"nan count: {nan_count}")

# Do any wafers repeat?
unique_values = np.unique(maps) # Expected: [0, 1, 2, 3]
# print(f"unique values: {unique_values}")

# Do defects appear in expected ratios?
for i, defect in enumerate(DEFECTS):
    count = (labels[:, i] != 0).sum()
    percent = (count / len(labels)) * 100
    # print(f"{defect}: {count:,} ({percent:.2f}%)")
